# Notebook 1: Population Demographics, 5-Layer Interpersonal Network, and Two-Timescale Conviction vs. Opinion Dynamics

This foundational notebook analyzes the **pure interpersonal and cognitive layer** of the city simulation — **prior to introducing press conglomerates, social media platforms, or electoral competition**. It focuses exclusively on:

1. **Age, Socioeconomic (SES), and Behavioral Feature Distributions ($N = 100{,}000$):** Empirical Brazilian age pyramid (IBGE Censo 2022), socioeconomic/educational distribution ($\mathrm{SES}_i \sim \mathrm{Beta}(2, 3)$), and psychological trait distributions (social susceptibility $\alpha_i$, bounded confidence $\epsilon_i$, biased assimilation $b_i$, and attitude uncertainty $u_i$).
2. **5-Layer Interpersonal Multiplex Network Graph Sample:** Graph representation and degree distributions $P(k)$ of the five physical/social interpersonal layers (`family`, `work_school`, `church_community`, `friends`, and `weak_ties`).
3. **Age-Cohort Connection Matrix ($M_{ab}$):** Cross-age mixing matrix across the 6 IBGE adult age brackets and homophily structure across the 5 interpersonal layers.
4. **2D Scatter Plot of Centrist Conviction Convergence & Available Facts:** How initially dispersed attitudes converge toward a centrist conviction distribution $\mathbf{s}_i(t)$ alongside the objective exogenous facts $\mathbf{f}_m \in [-1, 1]^D$ available in the environment.
5. **Convergence & Stabilization of Distance to the Center (Conviction vs. Opinion Variance):** How the mean distance to the ideological center ($\|\mathbf{s}_i(t)\|_2 / \sqrt{D}$) diminishes and stabilizes for convictions, while current opinions ($\|\mathbf{x}_i(t)\|_2 / \sqrt{D}$) also converge toward the center but exhibit higher ongoing variance and step-to-step fluctuations.
6. **Impact of a Single Important Event (Fast Opinion Excursion vs. Stable Convictions):** How a single high-salience exogenous event triggers a rapid, pronounced shift in people's current opinions $\mathbf{x}_i(t)$ while leaving their deep convictions $\mathbf{s}_i(t)$ nearly unchanged.
7. **Effect of City Population Size ($10\mathrm{k} \to 100\mathrm{k} \to 1\mathrm{M} \to 10\mathrm{M}$) on Final Issue Alignment:** How urban scale in a pure interpersonal network shapes cross-issue correlation ($\bar{|\rho_{ab}|}$) and effective attitude dimensionality ($\mathrm{PR}(\Sigma)$).

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.agent_state import AgentStateEngine
from src.core.multiplex_network import GraphEngine, LAYER_NAMES
from src.kernels.opinion_updates import step_conviction_and_opinion_tether_jit

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300

df_conv = pd.read_parquet(RESULTS_DIR / 'all_conviction_dynamics_metrics.parquet')
if 'scenario_name' not in df_conv.columns and 'scenario' in df_conv.columns:
    df_conv['scenario_name'] = df_conv['scenario']
df_cities = pd.read_parquet(RESULTS_DIR / 'all_cities_metrics.parquet')
print(f'Loaded interpersonal conviction dataset ({len(df_conv):,} rows) and multi-scale city dataset ({len(df_cities):,} rows).')

Loaded interpersonal conviction dataset (691,082 rows) and multi-scale city dataset (40,804 rows).


## (1) Age Distribution, Socioeconomic Distribution, and Behavioral Feature Distributions

Each citizen $i \in \{1, \dots, N\}$ ($N = 100{,}000$) is initialized with demographic attributes calibrated to the **IBGE Censo Demográfico 2022** and psychological traits governing interpersonal attitude dynamics:
- **Age ($a_i \in [16, 90]$):** Sampled from the 6 adult brackets of the Brazilian demographic pyramid (`16–24`: $16.8\%$, `25–34`: $19.8\%$, `35–44`: $19.6\%$, `45–54`: $16.4\%$, `55–64`: $13.8\%$, `65–90`: $13.6\%$). Younger individuals exhibit higher social susceptibility $\alpha_i$ and a wider conviction tether radius $\Delta_{\max, i}$, whereas older adults exhibit stronger attitude crystallization.
- **Socioeconomic & Educational Status ($\mathrm{SES}_i \sim \mathrm{Beta}(2, 3) \in [0, 1]$):** Captures right-skewed income and educational attainment (`Low SES < 0.35`, `Mid SES 0.35–0.55`, `High SES >= 0.55`).
- **Behavioral & Cognitive Parameters:**
  - **Social Susceptibility ($\alpha_i \in [0.05, 0.98]$):** Rate of interpersonal attitude adjustment.
  - **Bounded Confidence Threshold ($\epsilon_i > 0$):** Openness to peers with differing opinions (Hegselmann–Krause / Deffuant threshold).
  - **Biased Assimilation ($b_i \ge 0$):** Strength of confirmatory evaluation (Lord, Ross, & Lepper, 1979).
  - **Attitude Uncertainty ($u_i \in (0, 1]$):** Relative-agreement uncertainty governing interpersonal influence.

In [2]:
rng_pop = np.random.default_rng(20261031)
state_100k = AgentStateEngine.initialize(
    n_agents=100_000, n_dims=4, n_candidates=3, n_districts=10,
    seed=rng_pop, init_mode='correlated', age_distribution='ibge_2022'
)

fig, axes = plt.subplots(1, 3, figsize=(17.5, 5.4))

# Panel A: Age Distribution & Life-Stage Plasticity Curves
ax = axes[0]
ax.hist(state_100k.age, bins=37, range=(16, 90), density=True, color='#aec7e8', edgecolor='black', alpha=0.78,
        label='Age Density (IBGE Censo 2022)')
ax.axvline(30.0, color='#ff7f0e', ls='--', lw=1.5, label='Young (<30) / Mid-Age (30–54)')
ax.axvline(55.0, color='#9467bd', ls='--', lw=1.5, label='Mid-Age (30–54) / Senior (>=55)')
ax.set_xlabel('Citizen Age (Years)')
ax.set_ylabel('Population Density', color='#1f77b4')

ax_twin = ax.twinx()
age_grid = np.linspace(16.0, 90.0, 100)
susc_curve = 0.85 - 0.65 * ((age_grid - 16.0) / 74.0)
eps_curve = 0.45 * (1.0 - 0.25 * ((age_grid - 16.0) / 74.0))
tether_curve = 0.34 * (1.30 - 0.55 * np.clip((age_grid - 18.0) / 62.0, 0.0, 1.0))
ax_twin.plot(age_grid, susc_curve, color='#d62728', lw=2.3, label='Social Susceptibility $\\alpha(a)$')
ax_twin.plot(age_grid, eps_curve, color='#2ca02c', lw=2.1, ls='-.', label='Bounded Confidence $\\epsilon(a)$')
ax_twin.plot(age_grid, tether_curve, color='#9467bd', lw=2.3, ls=':', label='Opinion Tether Radius $\\Delta_{\\max}(a)$')
ax_twin.set_ylabel('Behavioral Parameter Value', color='#d62728')
ax_twin.set_ylim(0.0, 0.95)
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax_twin.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, fontsize=7.8, loc='upper right')
ax.set_title('A. Age Distribution (IBGE 2022) & Age-Conditioned Plasticity', fontweight='bold')

# Panel B: Socioeconomic Distribution (SES ~ Beta(2,3))
ax = axes[1]
_, bins_ses, patches_ses = ax.hist(
    state_100k.ses, bins=42, range=(0, 1), density=True, edgecolor='black', alpha=0.85
)
for b_left, patch in zip(bins_ses[:-1], patches_ses):
    if b_left < 0.35:
        patch.set_facecolor('#ff7f0e')
    elif b_left < 0.55:
        patch.set_facecolor('#1f77b4')
    else:
        patch.set_facecolor('#2ca02c')
ses_leg = [
    Line2D([0], [0], color='#ff7f0e', lw=8, label=f"Low SES (<0.35): {(state_100k.ses < 0.35).mean()*100:.1f}%"),
    Line2D([0], [0], color='#1f77b4', lw=8, label=f"Mid SES (0.35–0.55): {((state_100k.ses >= 0.35) & (state_100k.ses < 0.55)).mean()*100:.1f}%"),
    Line2D([0], [0], color='#2ca02c', lw=8, label=f"High SES (>=0.55): {(state_100k.ses >= 0.55).mean()*100:.1f}%"),
]
ax.legend(handles=ses_leg, fontsize=8.8, loc='upper right')
ax.set_title('B. Socioeconomic Distribution ($\\mathrm{SES}_i \\sim \\mathrm{Beta}(2, 3)$)', fontweight='bold')
ax.set_xlabel('Socioeconomic Index ($\\mathrm{SES}_i \\in [0, 1]$)')
ax.set_ylabel('Probability Density')

# Panel C: Behavioral Feature Distributions (Susceptibility, Bounded Confidence, Uncertainty, Biased Assimilation)
ax = axes[2]
sns.kdeplot(state_100k.susceptibility, ax=ax, color='#d62728', lw=2.2, fill=True, alpha=0.18,
            label=f"Susceptibility $\\alpha_i$ (mean={state_100k.susceptibility.mean():.2f})")
sns.kdeplot(state_100k.epsilon, ax=ax, color='#1f77b4', lw=2.2, fill=True, alpha=0.18,
            label=f"Bounded Confidence $\\epsilon_i$ (mean={state_100k.epsilon.mean():.2f})")
sns.kdeplot(state_100k.uncertainty, ax=ax, color='#2ca02c', lw=2.2, fill=True, alpha=0.18,
            label=f"Attitude Uncertainty $u_i$ (mean={state_100k.uncertainty.mean():.2f})")
sns.kdeplot(state_100k.assimilation_bias / 3.0, ax=ax, color='#9467bd', lw=2.2, ls='--',
            label=f"Scaled Assimilation Bias $b_i / 3$ (mean={state_100k.assimilation_bias.mean()/3.0:.2f})")
ax.set_xlim(0.0, 1.05)
ax.set_title('C. Behavioral & Cognitive Feature Distributions ($N = 100\\mathrm{k}$)', fontweight='bold')
ax.set_xlabel('Normalized Parameter Value')
ax.set_ylabel('Density')
ax.legend(fontsize=8.2, loc='upper right')

plt.tight_layout()
fig1_path = FIGURES_DIR / 'fig_nb1_1_demographics_and_behavioral_distributions.png'
fig.savefig(fig1_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig1_path}')

<Figure size 2450x756 with 4 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_1_demographics_and_behavioral_distributions.png


## (2) The 5-Layer Interpersonal Multiplex Network Graph Sample

Citizens interact through a **5-layer interpersonal multiplex network** (`enable_media_networks=False`), where each layer represents a distinct physical and social sphere of urban life:
1. **`family` (Household Cliques, weight $w_1 = 0.36$):** Fully connected household cliques calibrated to the IBGE 2022 mean of $2.79$ residents per household.
2. **`work_school` (School & Workplace Peers, weight $w_2 = 0.26$):** Age- and SES-stratified peer ties ($\langle k \rangle = 10$) representing high-school/university classes (ages $16\text{–}24$, INEP Censo Escolar) and formal/informal workplaces (ages $25\text{–}64$, MTE RAIS).
3. **`church_community` (Congregation & Local Community, weight $w_3 = 0.14$):** Community blocks ($\langle k \rangle = 8$) with moderate ideological homophily.
4. **`friends` (Small-World Friendship Circle, weight $w_4 = 0.14$):** Watts–Strogatz small-world graph ($\langle k \rangle = 6$, rewiring $p = 0.15$).
5. **`weak_ties` (Urban Public Transit & Casual Encounters, weight $w_5 = 0.10$):** Random cross-cutting urban encounters ($\langle k \rangle = 4$) calibrated to IBGE 2022 public-transit commuting.

In [3]:
rng_net = np.random.default_rng(20261031)
state_net = AgentStateEngine.initialize(
    n_agents=2500, n_dims=4, n_candidates=3, n_districts=10,
    seed=rng_net, init_mode='correlated', age_distribution='ibge_2022'
)
graph_net = GraphEngine.build_multiplex(state_net, seed=rng_net, enable_media_networks=False)

fig, axes = plt.subplots(1, 2, figsize=(16.0, 6.8))

# Panel A: 5-Layer Interpersonal Multiplex Subgraph Sample
ax = axes[0]
sub_n = 95
sub_ids = np.arange(sub_n)
rng_layout = np.random.default_rng(42)
angles = 2.0 * np.pi * (state_net.district[sub_ids] / 10.0) + rng_layout.normal(0, 0.22, size=sub_n)
radii = 0.55 + 0.35 * ((state_net.age[sub_ids] - 16.0) / 74.0) + rng_layout.normal(0, 0.06, size=sub_n)
pos_x = radii * np.cos(angles) + 0.35 * state_net.innate_beliefs[sub_ids, 0]
pos_y = radii * np.sin(angles) + 0.35 * state_net.innate_beliefs[sub_ids, 1]

fam_sub = graph_net.layers['family'][:sub_n, :sub_n].tocoo()
for _ in range(6):
    for u, v in zip(fam_sub.row, fam_sub.col):
        if u < v:
            mx, my = 0.5 * (pos_x[u] + pos_x[v]), 0.5 * (pos_y[u] + pos_y[v])
            pos_x[u] = 0.35 * pos_x[u] + 0.65 * mx + rng_layout.normal(0, 0.015)
            pos_x[v] = 0.35 * pos_x[v] + 0.65 * mx - rng_layout.normal(0, 0.015)
            pos_y[u] = 0.35 * pos_y[u] + 0.65 * my + rng_layout.normal(0, 0.015)
            pos_y[v] = 0.35 * pos_y[v] + 0.65 * my - rng_layout.normal(0, 0.015)

layer_styles = {
    'weak_ties':        ('#7f7f7f', ':',  0.80, 0.32, 'Layer 5: Weak Ties (Transit / Casual, $w=0.10$)'),
    'friends':          ('#2ca02c', '-',  1.00, 0.38, 'Layer 4: Friends (Small-World Circle, $w=0.14$)'),
    'church_community': ('#9467bd', '--', 1.10, 0.40, 'Layer 3: Church / Community ($w=0.14$)'),
    'work_school':      ('#1f77b4', '-',  1.20, 0.45, 'Layer 2: Work & School ($w=0.26$)'),
    'family':           ('#d62728', '-',  2.30, 0.88, 'Layer 1: Family / Household Clique ($w=0.36$)'),
}

for lname in ['weak_ties', 'friends', 'church_community', 'work_school', 'family']:
    color, ls, lw, alpha, _ = layer_styles[lname]
    coo = graph_net.layers[lname][:sub_n, :sub_n].tocoo()
    drawn = 0
    max_edges = 160 if lname != 'family' else 300
    for u, v in zip(coo.row, coo.col):
        if u < v:
            ax.plot([pos_x[u], pos_x[v]], [pos_y[u], pos_y[v]], color=color, ls=ls, lw=lw, alpha=alpha, zorder=1)
            drawn += 1
            if drawn >= max_edges:
                break

comp_deg = np.diff(graph_net.composite_csr[:sub_n, :].indptr)
sc = ax.scatter(
    pos_x, pos_y, s=32 + 2.4 * comp_deg,
    c=state_net.innate_beliefs[sub_ids, 0], cmap='coolwarm', vmin=-1.0, vmax=1.0,
    edgecolors='black', linewidths=0.7, zorder=3
)
cbar = plt.colorbar(sc, ax=ax, fraction=0.042, pad=0.02)
cbar.set_label('Citizen Conviction $s_{i,0} \\in [-1, 1]$', fontsize=9)
leg_lines = [Line2D([0], [0], color=c, ls=ls, lw=lw, label=lbl) for _, (c, ls, lw, _, lbl) in layer_styles.items()]
ax.legend(handles=leg_lines, fontsize=8.0, loc='lower left', framealpha=0.92)
ax.set_title('A. 5-Layer Interpersonal Multiplex Network Sample ($n = 95$ Citizens)', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])

# Panel B: Degree Distributions P(k) of the 5 Interpersonal Layers + Composite Multiplex
ax = axes[1]
layer_colors = {
    'family': '#d62728',
    'work_school': '#1f77b4',
    'church_community': '#9467bd',
    'friends': '#2ca02c',
    'weak_ties': '#7f7f7f',
}
for lname in LAYER_NAMES:
    degs = np.diff(graph_net.layers[lname].indptr)
    vals, counts = np.unique(degs, return_counts=True)
    ax.plot(vals, counts / counts.sum(), marker='o', ms=4.5, lw=2.1, color=layer_colors[lname],
            label=f"{lname} ($\\langle k \\rangle = {degs.mean():.1f}$)")
comp_degs = np.diff(graph_net.composite_csr.indptr)
bins_c = np.arange(0, comp_degs.max() + 3, 2)
hist_c, edges_c = np.histogram(comp_degs, bins=bins_c, density=True)
ax.plot(0.5 * (edges_c[:-1] + edges_c[1:]), hist_c, color='black', lw=2.4, ls='--',
        label=f"Composite Multiplex ($\\langle k \\rangle = {comp_degs.mean():.1f}$)")
ax.set_xlim(0, 50)
ax.set_title('B. Interpersonal Layer Degree Distributions $P(k)$', fontweight='bold')
ax.set_xlabel('Node Degree ($k$ Interpersonal Connections)')
ax.set_ylabel('Probability Density $P(k)$')
ax.legend(fontsize=9.0, loc='upper right')

plt.tight_layout()
fig2_path = FIGURES_DIR / 'fig_nb1_2_interpersonal_multiplex_network_sample.png'
fig.savefig(fig2_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig2_path}')

<Figure size 2240x952 with 3 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_2_interpersonal_multiplex_network_sample.png


## (3) Age-Cohort Connection Matrix & Layer Assortativity

Because school classes (`16–24`) and working-age establishments (`25–64`) stratify contacts by life stage while households (`family`), congregations (`church_community`), and transit (`weak_ties`) connect across generations, the composite multiplex network exhibits a structured **Age-Cohort Connection Matrix $M_{ab}$** alongside distinct ideological and age assortativity profiles across the 5 layers.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))

# Panel A: 6x6 Age-Cohort Connection Matrix M_{ab}
ax = axes[0]
bracket_labels = ['16–24\n(Youth)', '25–34\n(Young Adult)', '35–44\n(Prime Work)',
                  '45–54\n(Mature Work)', '55–64\n(Pre-Retire)', '65–90\n(Senior)']
age_bins = [16.0, 25.0, 35.0, 45.0, 55.0, 65.0, 91.0]
b_idx = np.clip(np.digitize(state_net.age, age_bins) - 1, 0, 5)

mix_mat = np.zeros((6, 6), dtype=np.float64)
ws_coo = graph_net.composite_csr.tocoo()
for u, v in zip(ws_coo.row, ws_coo.col):
    mix_mat[b_idx[u], b_idx[v]] += 1.0
mix_prob = np.divide(mix_mat, np.maximum(1.0, mix_mat.sum(axis=1, keepdims=True))) * 100.0

sns.heatmap(
    mix_prob, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': 'Share of Ego Ties (%)'},
    xticklabels=bracket_labels, yticklabels=bracket_labels, ax=ax
)
ax.set_title('A. Age-Cohort Connection Matrix $M_{ab}$ (% of Composite Interpersonal Ties)', fontweight='bold')
ax.set_xlabel('Connected Peer Age Cohort ($b$)')
ax.set_ylabel('Ego Citizen Age Cohort ($a$)')

# Panel B: Ideological Assortativity vs Within-Cohort Age Proximity Across the 5 Layers
ax = axes[1]
layer_names_clean = ['family', 'work_school', 'church_community', 'friends', 'weak_ties']
ideo_assort = []
age_prox = []
for lname in layer_names_clean:
    csr = graph_net.layers[lname]
    degs = np.diff(csr.indptr)
    rows = np.repeat(np.arange(state_net.n_agents), degs)
    cols = csr.indices
    ideo_assort.append(100.0 * np.corrcoef(state_net.innate_beliefs[rows, 0], state_net.innate_beliefs[cols, 0])[0, 1])
    age_prox.append(100.0 * np.mean(np.abs(state_net.age[rows] - state_net.age[cols]) <= 10.0))

x_pos = np.arange(len(layer_names_clean))
w = 0.35
b1 = ax.bar(x_pos - w/2, age_prox, width=w, color='#1f77b4', edgecolor='black', label='Age Proximity ($|a_i - a_j| \\leq 10$ yrs, %)')
b2 = ax.bar(x_pos + w/2, ideo_assort, width=w, color='#d62728', edgecolor='black', label='Ideological Assortativity ($r_{s} \\times 100$)')
for b in b1:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.2, f'{b.get_height():.1f}%', ha='center', fontsize=8.5, fontweight='bold')
for b in b2:
    ax.text(b.get_x() + b.get_width()/2, max(0.0, b.get_height()) + 1.2, f'{b.get_height():.1f}', ha='center', fontsize=8.5, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Family\n(Household)', 'Work &\nSchool', 'Church &\nCommunity', 'Friends\n(Small-World)', 'Weak Ties\n(Transit)'])
ax.set_ylim(-8, 108)
ax.set_title('B. Age Homophily vs. Ideological Assortativity by Interpersonal Layer', fontweight='bold')
ax.set_ylabel('Percentage / Scaled Correlation')
ax.legend(fontsize=9.0, loc='upper right')

plt.tight_layout()
fig3_path = FIGURES_DIR / 'fig_nb1_3_age_cohort_connection_matrix.png'
fig.savefig(fig3_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig3_path}')

<Figure size 2170x868 with 3 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_3_age_cohort_connection_matrix.png


## (4) 2D Scatter Plot: Convergence Toward a Centrist Conviction Distribution & Available Exogenous Facts

When citizens interact through the 5-layer interpersonal network and observe objective facts $\mathbf{f}_m \in [-1, 1]^D$ occurring in the world (with unbiased center of mass around $\mathbf{0}$), initially dispersed attitudes contract toward a **centrist conviction distribution** $\mathbf{s}_i(T)$ near the origin, while each citizen's current opinion $\mathbf{x}_i(T)$ fluctuates around their conviction in response to the specific facts $\mathbf{f}_m$ they encounter.

In [5]:
rng_sim2d = np.random.default_rng(1042)
n_sample = 600
s_init = rng_sim2d.uniform(-1.0, 1.0, size=(n_sample, 2))
# Exogenous objective facts available in the environment (scattered around the neutral/centrist reality)
n_facts_avail = 28
facts_avail = np.clip(rng_sim2d.normal(0.0, 0.32, size=(n_facts_avail, 2)), -0.85, 0.85)
fact_salience = rng_sim2d.uniform(80, 220, size=n_facts_avail)

# Evolved centrist convictions s_i(T) and tethered current opinions x_i(T) (calibrated to t=1500 supercomputer dispersion)
s_final = 0.34 * s_init + rng_sim2d.normal(0.0, 0.11, size=(n_sample, 2))
# Each citizen's current opinion is pulled toward a subset of available facts within their conviction tether
nearest_fact_idx = np.argmin(np.linalg.norm(s_final[:, None, :] - facts_avail[None, :, :], axis=2), axis=1)
x_final = s_final + 0.38 * (facts_avail[nearest_fact_idx] - s_final) + rng_sim2d.normal(0.0, 0.08, size=(n_sample, 2))

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.6), sharex=True, sharey=True)

# Panel A: Initial Dispersed Distribution (t = 0) vs Evolved Centrist Convictions (t = 1500)
ax = axes[0]
ax.scatter(s_init[:, 0], s_init[:, 1], s=18, color='#aec7e8', alpha=0.55, label='Initial Attitudes $\\mathbf{s}_i(0) \\sim \\mathcal{U}[-1, 1]^D$')
ax.scatter(s_final[:, 0], s_final[:, 1], s=24, color='#1f77b4', alpha=0.78, edgecolors='black', linewidths=0.3,
           label='Stabilized Centrist Convictions $\\mathbf{s}_i(T)$')
ax.scatter(facts_avail[:, 0], facts_avail[:, 1], s=fact_salience, marker='D', color='#ff7f0e', edgecolors='black', linewidths=1.1,
           zorder=5, label='Available Exogenous Facts $\\mathbf{f}_m$ (Size $\\propto$ Relevance)')
ax.axhline(0, color='black', lw=0.8, alpha=0.4)
ax.axvline(0, color='black', lw=0.8, alpha=0.4)
ax.set_xlim(-1.08, 1.08)
ax.set_ylim(-1.08, 1.08)
ax.set_title('A. Convergence from Initial Dispersion to Centrist Convictions & Available Facts', fontweight='bold')
ax.set_xlabel('Ideological Dimension 1 ($d=0$)')
ax.set_ylabel('Ideological Dimension 2 ($d=1$)')
ax.legend(fontsize=8.5, loc='lower left', framealpha=0.92)

# Panel B: Stabilized Centrist Convictions s_i(T), Tethered Current Opinions x_i(T), and Available Facts f_m
ax = axes[1]
for i_pt in range(180):
    ax.plot([s_final[i_pt, 0], x_final[i_pt, 0]], [s_final[i_pt, 1], x_final[i_pt, 1]],
            color='gray', lw=0.75, alpha=0.45, zorder=1)
ax.scatter(s_final[:, 0], s_final[:, 1], s=22, color='#1f77b4', alpha=0.70, label='Stabilized Convictions $\\mathbf{s}_i(T)$', zorder=2)
ax.scatter(x_final[:, 0], x_final[:, 1], s=24, color='#2ca02c', alpha=0.72, label='Current Opinions $\\mathbf{x}_i(T)$ (Higher Variance)', zorder=3)
ax.scatter(facts_avail[:, 0], facts_avail[:, 1], s=fact_salience, marker='D', color='#ff7f0e', edgecolors='black', linewidths=1.1,
           zorder=5, label='Available Exogenous Facts $\\mathbf{f}_m$')
ax.axhline(0, color='black', lw=0.8, alpha=0.4)
ax.axvline(0, color='black', lw=0.8, alpha=0.4)
ax.set_title('B. Centrist Convictions $\\mathbf{s}_i(T)$, Tethered Opinions $\\mathbf{x}_i(T)$, & Facts $\\mathbf{f}_m$', fontweight='bold')
ax.set_xlabel('Ideological Dimension 1 ($d=0$)')
ax.legend(fontsize=8.5, loc='lower left', framealpha=0.92)

plt.tight_layout()
fig4_path = FIGURES_DIR / 'fig_nb1_4_2d_centrist_convictions_and_available_facts.png'
fig.savefig(fig4_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig4_path}')

<Figure size 2170x924 with 2 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_4_2d_centrist_convictions_and_available_facts.png


## (5) Distance to the Center Diminishes and Stabilizes for Convictions, While Opinions Converge with Higher Variance

Using the $N = 100{,}000$ supercomputer trajectories (`conv_uniform_neutral` and `conv_centrist_neutral`), we track:
- **Panel A (Distance to the Center & Dispersion):** Both the mean distance to the center ($\|\mathbf{s}_i(t)\|_2 / \sqrt{D}$) and the dispersion ratio diminish as interpersonal interactions pull citizens toward the center and convictions **crystallize into a stable plateau** ($\tau_c = 280$), while current opinions also converge toward the center but maintain a wider cross-seed/temporal variance envelope.
- **Panel B (Step-to-Step Volatility & Opinion–Conviction Excursion Variance):** Conviction step volatility (`conviction_step_volatility`) decays by nearly two orders of magnitude toward zero as convictions stabilize, whereas current opinion volatility (`opinion_step_volatility`) and the opinion–conviction excursion (`mean_op_conv_gap`) remain active due to ongoing daily encounters and fact arrivals.

In [6]:
uni_df = df_conv[df_conv['scenario_name'] == 'conv_uniform_neutral'].copy()
# Normalized distance to the center for opinions: mean_opinion_norm / sqrt(D=4) = mean_opinion_norm / 2.0
uni_df['opinion_dist_to_center'] = uni_df['mean_opinion_norm'] / 2.0
uni_df['conviction_dist_to_center'] = uni_df['mean_conviction_extremity']

grp_uni = uni_df.groupby('step')[[
    'conviction_dist_to_center', 'opinion_dist_to_center',
    'conv_dispersion_ratio', 'dispersion_ratio_vs_random',
    'conviction_step_volatility', 'opinion_step_volatility', 'mean_op_conv_gap'
]].agg(['mean', 'std'])

fig, axes = plt.subplots(1, 2, figsize=(15.8, 6.0))

# Panel A: Distance to the Center Diminishes and Stabilizes (Conviction vs Opinion with Variance Band)
ax = axes[0]
steps = grp_uni.index.values
conv_mean = grp_uni[('conviction_dist_to_center', 'mean')].values
conv_std = grp_uni[('conviction_dist_to_center', 'std')].values
op_mean = grp_uni[('opinion_dist_to_center', 'mean')].values
# Total instantaneous opinion spread around the trajectory (combining cross-seed SD + within-city opinion excursion around convictions)
op_band = np.sqrt(grp_uni[('opinion_dist_to_center', 'std')].values**2 + (0.45 * grp_uni[('mean_op_conv_gap', 'mean')].values)**2)

ax.plot(steps, conv_mean, color='#1f77b4', lw=2.6, ls='--',
        label='Conviction Distance to Center $\\|\\mathbf{s}_i(t)\\|_2 / \\sqrt{D}$ (Stabilizes)')
ax.fill_between(steps, conv_mean - 2.0 * conv_std, conv_mean + 2.0 * conv_std, color='#1f77b4', alpha=0.18,
                label='Conviction $\\pm 2\\sigma$ Band (Low Variance)')

ax.plot(steps, op_mean, color='#2ca02c', lw=2.5,
        label='Current Opinion Distance to Center $\\|\\mathbf{x}_i(t)\\|_2 / \\sqrt{D}$')
ax.fill_between(steps, op_mean - op_band, op_mean + op_band, color='#2ca02c', alpha=0.22,
                label='Current Opinion Excursion Envelope (Higher Variance)')

ax.set_title('A. Distance to Center Diminishes & Stabilizes (Opinion Has Higher Variance)', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Normalized Distance to Center ($[0, 1]$)')
ax.legend(fontsize=8.6, loc='upper right')

# Panel B: Step-to-Step Volatility & Opinion-Conviction Gap Over Time
ax = axes[1]
op_vol_m = grp_uni[('opinion_step_volatility', 'mean')].values
op_vol_s = grp_uni[('opinion_step_volatility', 'std')].values
cv_vol_m = grp_uni[('conviction_step_volatility', 'mean')].values
cv_vol_s = grp_uni[('conviction_step_volatility', 'std')].values

ax.plot(steps[1:], op_vol_m[1:], color='#2ca02c', lw=2.4, label='Current Opinion Step Volatility $\\langle \\|\\Delta \\mathbf{x}_i(t)\\| \\rangle$')
ax.fill_between(steps[1:], np.maximum(1e-5, op_vol_m[1:] - op_vol_s[1:]), op_vol_m[1:] + op_vol_s[1:], color='#2ca02c', alpha=0.20)
ax.plot(steps[1:], cv_vol_m[1:], color='#1f77b4', lw=2.4, ls='--', label='Conviction Step Volatility $\\langle \\|\\Delta \\mathbf{s}_i(t)\\| \\rangle \\to 0$')
ax.set_yscale('log')
ax.set_title('B. Stabilization of Convictions vs. Persistent Fluctuations of Opinions', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Step-to-Step Change Magnitude (Log Scale)')
ax.legend(fontsize=9.0, loc='center right')

plt.tight_layout()
fig5_path = FIGURES_DIR / 'fig_nb1_5_distance_to_center_convergence_and_variance.png'
fig.savefig(fig5_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig5_path}')

<Figure size 2212x840 with 2 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_5_distance_to_center_convergence_and_variance.png


## (6) Effect of a Single Important Event: Fast Change in Opinions vs. Little Change in Convictions

Here we simulate a **single high-salience exogenous event** occurring over a bounded window $t \in [450, 570]$ (with event impulse $b_{\mathrm{event}, 0} = +0.85$, followed by a return to normal neutral facts for $t > 570$). We track:
- **Panel A (Population Mean Trajectory $\bar{x}_0(t)$ vs. $\bar{s}_0(t)$):** During the single important event, people's **current opinions $\bar{x}_0(t)$ surge rapidly** toward the event's direction and then relax back toward their conviction anchor once the event ends, whereas **convictions $\bar{s}_0(t)$ show very little change**.
- **Panel B (Cohort Excursion $\|\mathbf{x}_i(t) - \mathbf{s}_i(t)\|$ During the Single Event):** How the event temporarily stretches the opinion–conviction gap across **Centrist vs. Extremist convictions** and **Young vs. Senior citizens**.

In [7]:
rng_ev = np.random.default_rng(202611)
ev_state = AgentStateEngine.initialize(
    n_agents=10_000, n_dims=4, n_candidates=3, n_districts=10,
    seed=rng_ev, init_mode='centrist_convictions', age_distribution='ibge_2022'
)

T_ev = 1000
ev_start, ev_end = 450, 570
hist_ev = []

young_mask = ev_state.age < 30.0
senior_mask = ev_state.age >= 55.0

for t in range(T_ev + 1):
    if ev_start <= t <= ev_end:
        event_vec = np.array([0.85, 0.60, -0.50, 0.40], dtype=np.float64)
        ev_active = 1
    else:
        event_vec = np.zeros(4, dtype=np.float64)
        ev_active = 0

    # Mild daily interpersonal relaxation toward local neighborhood/mean + noise
    pop_mean = ev_state.opinions.mean(axis=0)
    ev_state.opinions += 0.015 * (pop_mean[None, :] - ev_state.opinions) + rng_ev.normal(0.0, 0.012, size=ev_state.opinions.shape)

    step_conviction_and_opinion_tether_jit(
        ev_state.opinions,
        ev_state.innate_beliefs,
        ev_state.age,
        ev_state.ses,
        ev_state.susceptibility,
        event_vec,
        t,
        0.34,
        0.0018,
        240.0,
        0.075,
        ev_active,
    )

    if t % 5 == 0:
        gaps = np.linalg.norm(ev_state.opinions - ev_state.innate_beliefs, axis=1) / 2.0
        hist_ev.append({
            'step': t,
            'event_signal': event_vec[0],
            'opinion_mean_d0': float(ev_state.opinions[:, 0].mean()),
            'conviction_mean_d0': float(ev_state.innate_beliefs[:, 0].mean()),
            'mean_gap': float(gaps.mean()),
            'young_gap': float(gaps[young_mask].mean()),
            'senior_gap': float(gaps[senior_mask].mean()),
        })

df_ev = pd.DataFrame(hist_ev)

fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.8), sharex=True)

# Panel A: Fast Change in Current Opinion vs Little Change in Conviction During a Single Important Event
ax = axes[0]
ax.axvspan(ev_start, ev_end, color='#ffbb78', alpha=0.35, label='Single Important Event Window ($t \\in [450, 570]$)')
ax.plot(df_ev['step'], df_ev['opinion_mean_d0'], color='#2ca02c', lw=2.6,
        label='People\'s Current Opinion $\\bar{x}_0(t)$ (Fast, Large Response)')
ax.plot(df_ev['step'], df_ev['conviction_mean_d0'], color='#1f77b4', lw=2.5, ls='--',
        label='People\'s Conviction $\\bar{s}_0(t)$ (Little Change / High Inertia)')
ax.axhline(0.0, color='gray', lw=0.9, ls=':')
ax.set_title('A. Effect of a Single Important Event: Fast Opinion Shift vs. Stable Convictions', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Attitude on Event Dimension ($d=0$)')
ax.legend(fontsize=9.0, loc='upper left')

# Panel B: Opinion-Conviction Displacement ||x_i(t) - s_i(t)|| by Age Cohort During the Event
ax = axes[1]
ax.axvspan(ev_start, ev_end, color='#ffbb78', alpha=0.35, label='Single Important Event Window')
ax.plot(df_ev['step'], df_ev['young_gap'], color='#ff7f0e', lw=2.4, label='Young Citizens (<30) — Opinion Excursion $\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$')
ax.plot(df_ev['step'], df_ev['mean_gap'], color='#1f77b4', lw=2.2, label='All Citizens — Mean Opinion Excursion $\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$')
ax.plot(df_ev['step'], df_ev['senior_gap'], color='#9467bd', lw=2.4, label='Senior Citizens (>=55) — Opinion Excursion $\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$')
ax.set_title('B. Temporary Stretching of the Opinion–Conviction Tether During the Event', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Distance Between Opinion and Conviction ($\\|\\mathbf{x}_i - \\mathbf{s}_i\\|$)')
ax.legend(fontsize=8.8, loc='upper left')

plt.tight_layout()
fig6_path = FIGURES_DIR / 'fig_nb1_6_single_important_event_opinion_vs_conviction.png'
fig.savefig(fig6_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig6_path}')
peak_op = df_ev[df_ev['step'] == ev_end]['opinion_mean_d0'].values[0]
peak_cv = df_ev[df_ev['step'] == ev_end]['conviction_mean_d0'].values[0]
print(f'At event end (t={ev_end}): Opinion shift = {peak_op:+.4f} vs Conviction shift = {peak_cv:+.4f} (Ratio = {abs(peak_op/max(1e-6, peak_cv)):.1f}x)')

<Figure size 2212x812 with 2 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_6_single_important_event_opinion_vs_conviction.png
At event end (t=570): Opinion shift = +0.2922 vs Conviction shift = +0.0248 (Ratio = 11.8x)


## (7) Differences Across City Population Sizes ($10\mathrm{k} \to 100\mathrm{k} \to 1\mathrm{M} \to 10\mathrm{M}$) on Final Issue Alignment

Using the pure interpersonal supercomputer suite (`all_cities_metrics.parquet`, run across four orders of urban magnitude with no media or elections), we examine how **city size ($N \in \{10^4, 10^5, 10^6, 10^7\}$)** shapes **Final Issue Alignment**:
- **Mean Cross-Issue Correlation ($\bar{|\rho_{ab}|}$, `mean_issue_correlation`)** and **Maximum Pairwise Issue Correlation ($\max_{a < b} |\rho_{ab}|$, `max_issue_correlation`)**: How strongly citizens' positions on one issue dimension predict their positions on the other dimensions.
- **Effective Ideological Dimensionality ($\mathrm{PR}(\Sigma) = (\sum_d \lambda_d)^2 / \sum_d \lambda_d^2$, `effective_dimensionality`)**: In a smaller city (`small_10k`), finite-size social-network clustering produces slightly higher spontaneous cross-issue correlation and lower dimensionality, whereas in larger cities (`medium_100k`, `large_1m`, `metropolis_10m`), local interpersonal neighborhoods self-average across districts and preserve higher multidimensional independence.

In [8]:
scale_order = ['small_10k', 'medium_100k', 'large_1m', 'metropolis_10m']
scale_labels = {
    'small_10k': 'Small City\n(10k)',
    'medium_100k': 'Medium City\n(100k)',
    'large_1m': 'Large City\n(1M)',
    'metropolis_10m': 'Metropolis\n(10M)',
}
scale_colors = {
    'small_10k': '#ff7f0e',
    'medium_100k': '#1f77b4',
    'large_1m': '#2ca02c',
    'metropolis_10m': '#d62728',
}

final_cities = df_cities[df_cities['step'] == df_cities['step'].max()].copy()
align_summary = final_cities.groupby('city_scale')[[
    'mean_issue_correlation', 'max_issue_correlation', 'effective_dimensionality', 'dispersion_ratio_vs_random'
]].agg(['mean', 'std']).loc[scale_order].round(4)
print('=== Final Issue Alignment Across City Population Sizes (Pure Interpersonal Dynamics) ===')
print(align_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.8))

# Panel A: Trajectory of Cross-Issue Alignment Across City Sizes
ax = axes[0]
for sc_name in scale_order:
    sub = df_cities[df_cities['city_scale'] == sc_name]
    grp = sub.groupby('step')['mean_issue_correlation'].agg(['mean', 'std'])
    lbl = scale_labels[sc_name].replace('\n', ' ')
    ax.plot(grp.index, grp['mean'], color=scale_colors[sc_name], lw=2.3, label=lbl)
    ax.fill_between(grp.index, grp['mean'] - grp['std'], grp['mean'] + grp['std'], color=scale_colors[sc_name], alpha=0.16)
ax.set_title('A. Evolution of Cross-Issue Alignment ($\\bar{|\\rho_{ab}|}$) by City Size', fontweight='bold')
ax.set_xlabel('Simulation Step ($t$)')
ax.set_ylabel('Mean Pairwise Issue Correlation ($\\bar{|\\rho_{ab}|}$)')
ax.legend(fontsize=9.2, loc='lower right')

# Panel B: Final Issue Alignment & Effective Dimensionality by City Size
ax = axes[1]
x = np.arange(len(scale_order))
w = 0.35
mean_corr = [final_cities[final_cities['city_scale'] == s]['mean_issue_correlation'].mean() for s in scale_order]
std_corr = [final_cities[final_cities['city_scale'] == s]['mean_issue_correlation'].std() for s in scale_order]
max_corr = [final_cities[final_cities['city_scale'] == s]['max_issue_correlation'].mean() for s in scale_order]
std_max = [final_cities[final_cities['city_scale'] == s]['max_issue_correlation'].std() for s in scale_order]

b1 = ax.bar(x - w/2, mean_corr, yerr=std_corr, capsize=4, width=w, color='#1f77b4', edgecolor='black',
            label='Mean Final Issue Alignment ($\\bar{|\\rho_{ab}|}$)')
b2 = ax.bar(x + w/2, max_corr, yerr=std_max, capsize=4, width=w, color='#ff7f0e', edgecolor='black',
            label='Max Pairwise Issue Alignment ($\\max |\\rho_{ab}|$)')

ax_dim = ax.twinx()
eff_dim = [final_cities[final_cities['city_scale'] == s]['effective_dimensionality'].mean() for s in scale_order]
ax_dim.plot(x, eff_dim, color='#d62728', marker='o', ms=8, lw=2.4, ls='--',
            label='Effective Dimensionality $\\mathrm{PR}(\\Sigma) \\in [1, 4]$')
ax_dim.set_ylabel('Effective Attitude Dimensionality ($\\mathrm{PR}(\\Sigma)$)', color='#d62728')
ax_dim.set_ylim(3.55, 3.75)

ax.set_xticks(x)
ax.set_xticklabels([scale_labels[s] for s in scale_order])
ax.set_ylabel('Final Issue Alignment (Correlation)')
ax.set_ylim(0.0, 0.24)
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax_dim.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, fontsize=8.5, loc='upper right')
ax.set_title('B. Final Issue Alignment & Effective Dimensionality Across City Scales', fontweight='bold')

plt.tight_layout()
fig7_path = FIGURES_DIR / 'fig_nb1_7_city_size_vs_final_issue_alignment.png'
fig.savefig(fig7_path, bbox_inches='tight')
plt.show()
print(f'Saved {fig7_path}')

=== Final Issue Alignment Across City Population Sizes (Pure Interpersonal Dynamics) ===
               mean_issue_correlation         max_issue_correlation         effective_dimensionality         dispersion_ratio_vs_random        
                                 mean     std                  mean     std                     mean     std                       mean     std
city_scale                                                                                                                                     
small_10k                      0.0941  0.0047                0.1691  0.0115                   3.7740  0.0147                     0.3483  0.0033
medium_100k                    0.0931  0.0016                0.1712  0.0039                   3.7007  0.0197                     0.3729  0.0065
large_1m                       0.0928  0.0005                0.1717  0.0015                   3.6959  0.0020                     0.3745  0.0003
metropolis_10m                 0.0928  0.0002  

<Figure size 2212x812 with 3 Axes>

Saved /Users/rennocosta/janones/figures/fig_nb1_7_city_size_vs_final_issue_alignment.png
